# StyleMe — 00: Setup & Dataset Download

Everything lands under `STYLEME_ROOT` (default `/content/StyleMe` in Colab CLI; Google Drive when run in the browser).

**Runtime:** T4 GPU (free tier fine). No TPU needed — the pipeline is MobileCLIP + sklearn.

**Verified sources** (checked against live Kaggle/HF APIs):

| what | source | size | used by |
|---|---|---|---|
| DeepFashion2 val (images + bbox/category CSV) | `jami145/deepfashion2` | 7.5 GB zip, extracts ~0.6 GB | 01 slot eval |
| Fashion Product Images (small) | `paramaggarwal/fashion-product-images-small` | 0.6 GB | 01 color/formality eval |
| Fashionpedia clean annotations | `tpgd01/fashionpedia-filter-attributes` | 70 MB | 01 pattern/fit eval |
| Fashionpedia val images (parquet) | HF `detection-datasets/fashionpedia` | 85 MB | 01 pattern/fit eval |
| Polyvore outfit compatibility annotations | `nguynmcch/polyvore-outfits-data` | ~10 MB | 03 engine calibration |

**Survival:** all steps are idempotent (`.done` markers). Re-running after a disconnect skips finished downloads.

In [ ]:
!pip install -q kaggle huggingface_hub pandas pyarrow pyyaml

In [ ]:
import os, sys, json, glob, shutil, subprocess, zipfile, time

ROOT = os.environ.get('STYLEME_ROOT')
if not ROOT:
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_notify=20)
        ROOT = '/content/drive/MyDrive/StyleMe'
    except Exception:
        ROOT = '/content/StyleMe'
        print('note: Drive not mounted — using ephemeral /content/StyleMe')

RAW = os.path.join(ROOT, 'raw')
ART = os.path.join(ROOT, 'artifacts')
DL  = '/content/styleme_dl'          # scratch, never on Drive
os.makedirs(RAW, exist_ok=True)
os.makedirs(ART, exist_ok=True)
os.makedirs(DL, exist_ok=True)
print('ROOT:', ROOT)
print('scratch:', DL)

ROOT: /content/StyleMe
scratch: /content/styleme_dl


In [ ]:
import os

def kaggle_auth():
    tok = os.environ.get('KAGGLE_API_TOKEN')
    if not tok:
        p = os.path.expanduser('~/.kaggle/access_token')
        if os.path.exists(p):
            tok = open(p).read().strip()
            os.environ['KAGGLE_API_TOKEN'] = tok
    if not tok:
        p = os.path.expanduser('~/.kaggle/kaggle.json')
        if os.path.exists(p):
            os.environ['KAGGLE_CONFIG_DIR'] = os.path.dirname(p)
            return True
    return bool(tok)

KAGGLE_OK = kaggle_auth()
print('kaggle auth:', 'READY' if KAGGLE_OK else 'MISSING (public HF downloads still work)')

kaggle auth: READY


In [ ]:
def sh(cmd):
    print('$', ' '.join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
        raise RuntimeError(f'command failed: {cmd[:2]}')
    return r

def kaggle_dl(slug, dest, filename=None, unzip=True):
    assert KAGGLE_OK, 'no kaggle auth'
    args = ['kaggle', 'datasets', 'download', '-d', slug, '-p', dest]
    if filename:
        args += ['-f', filename]
    if unzip:
        args += ['--unzip']
    sh(args)

def done(name):
    return os.path.exists(os.path.join(ROOT, f'{name}.done'))

def mark(name, note=''):
    open(os.path.join(ROOT, f'{name}.done'), 'w').write(note or time.ctime())
    print(f'DONE {name}: {note}')

def extract_members(zpath, out_dir, predicate, rename=None):
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    with zipfile.ZipFile(zpath) as z:
        members = [i for i in z.infolist() if predicate(i.filename)]
        print(f'{len(members)} matching members in {os.path.basename(zpath)}', flush=True)
        for j, info in enumerate(members):
            name = os.path.basename(info.filename)
            if rename:
                name = rename(info.filename, name)
            if not name:
                continue
            target = os.path.join(out_dir, name)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            n += 1
            if n % 5000 == 0:
                print(f'  extracted {n}/{len(members)}', flush=True)
    return n

## 1. DeepFashion2 — validation split only

The Kaggle mirror ships a 7.5 GB zip containing train/test/val. We extract **only** `input/validation.csv` + `resized/validation/*.jpg`, then delete the zip. That leaves ~0.6 GB on disk.

In [ ]:
if done('deepfashion2'):
    print('already present — skip')
else:
    df2_dl = os.path.join(DL, 'df2'); os.makedirs(df2_dl, exist_ok=True)
    sh(['kaggle', 'datasets', 'download', '-d', 'jami145/deepfashion2', '-p', df2_dl])
    zips = glob.glob(os.path.join(df2_dl, '*.zip'))
    assert zips, 'no zip downloaded'
    zpath = zips[0]
    print('zip:', zpath, f'{os.path.getsize(zpath)/1e9:.1f} GB')

    out = os.path.join(RAW, 'deepfashion2')
    csv_n = extract_members(
        zpath, out,
        lambda n: n.endswith('input/validation.csv'),
        rename=lambda full, base: 'validation.csv')
    img_n = extract_members(
        zpath, os.path.join(out, 'images'),
        lambda n: '/resized/validation/' in n and n.lower().endswith(('.jpg', '.jpeg', '.png')))

    os.remove(zpath)
    shutil.rmtree(df2_dl, ignore_errors=True)
    mark('deepfashion2', f'{img_n} images, {csv_n} csv')

$ kaggle datasets download -d jami145/deepfashion2 -p /content/styleme_dl/df2


zip: /content/styleme_dl/df2/deepfashion2.zip 7.5 GB


1 matching members in deepfashion2.zip


32153 matching members in deepfashion2.zip


  extracted 5000/32153


  extracted 10000/32153


  extracted 15000/32153


  extracted 20000/32153


  extracted 25000/32153


  extracted 30000/32153


DONE deepfashion2: 32153 images, 1 csv


## 2. Fashion Product Images (small) — color + formality ground truth

44k product photos with `styles.csv` (`baseColour`, `usage`, `articleType`). Small images (60x80) are fine for color and coarse formality.

In [ ]:
if done('fashion-product'):
    print('already present — skip')
else:
    fp_dl = os.path.join(DL, 'fp'); os.makedirs(fp_dl, exist_ok=True)
    kaggle_dl('paramaggarwal/fashion-product-images-small', fp_dl, unzip=True)
    out = os.path.join(RAW, 'fashion-product')
    os.makedirs(out, exist_ok=True)
    src_imgs = os.path.join(fp_dl, 'images')
    src_csv  = os.path.join(fp_dl, 'styles.csv')
    assert os.path.isdir(src_imgs) and os.path.exists(src_csv), os.listdir(fp_dl)
    if not os.path.exists(os.path.join(out, 'images')):
        shutil.move(src_imgs, os.path.join(out, 'images'))
    shutil.move(src_csv, os.path.join(out, 'styles.csv'))
    shutil.rmtree(fp_dl, ignore_errors=True)
    mark('fashion-product', 'images/ + styles.csv')

$ kaggle datasets download -d paramaggarwal/fashion-product-images-small -p /content/styleme_dl/fp --unzip


DONE fashion-product: images/ + styles.csv


## 3. Fashionpedia — pattern + fit ground truth

Annotations carry `attribute_ids` (textile pattern, silhouette fit). Images come from the HF parquet mirror, matched by `image_id`.

In [ ]:
if done('fashionpedia'):
    print('already present — skip')
else:
    fpa_dl = os.path.join(DL, 'fpa'); os.makedirs(fpa_dl, exist_ok=True)
    kaggle_dl('tpgd01/fashionpedia-filter-attributes', fpa_dl, unzip=True)
    ann_dir = os.path.join(RAW, 'fashionpedia', 'annotations')
    os.makedirs(ann_dir, exist_ok=True)
    found = glob.glob(os.path.join(fpa_dl, '**', '*.json'), recursive=True)
    print('annotation files:', [os.path.basename(f) for f in found])
    for f in found:
        shutil.copy(f, os.path.join(ann_dir, os.path.basename(f)))
    shutil.rmtree(fpa_dl, ignore_errors=True)
    mark('fashionpedia-annotations', f'{len(found)} json files')

$ kaggle datasets download -d tpgd01/fashionpedia-filter-attributes -p /content/styleme_dl/fpa --unzip


annotation files: ['val.json', 'train.json', 'test.json']


DONE fashionpedia-annotations: 3 json files


In [ ]:
from huggingface_hub import hf_hub_download

HF_PARQUET = 'data/val-00000-of-00001-f915eb4a6ec8c97e.parquet'
fp_img_dir = os.path.join(RAW, 'fashionpedia', 'hf')
target = os.path.join(fp_img_dir, HF_PARQUET)
if os.path.exists(target):
    print('fashionpedia images already present — skip')
else:
    p = hf_hub_download(
        repo_id='sandrocaseiro/fashionpedia',
        filename=HF_PARQUET,
        repo_type='dataset',
        local_dir=fp_img_dir)
    print('downloaded:', p, f'{os.path.getsize(p)/1e6:.0f} MB')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


data/val-00000-of-00001-f915eb4a6ec8c97e(…): reconstructing file:   0%|          |  0.00B / 86.8MB            

data/val-00000-of-00001-f915eb4a6ec8c97e(…): downloading bytes:           |  0.00B            

downloaded: /content/StyleMe/raw/fashionpedia/hf/data/val-00000-of-00001-f915eb4a6ec8c97e.parquet 87 MB


## 4. Polyvore — outfit compatibility annotations (for notebook 03)

Only the small annotation files; no images needed.

In [ ]:
if done('polyvore'):
    print('already present — skip')
else:
    poly_dl = os.path.join(DL, 'poly'); os.makedirs(poly_dl, exist_ok=True)
    files = [
        'polyvore-outfits/categories.csv',
        'polyvore-outfits/disjoint/fill_in_blank_train.json',
        'polyvore-outfits/disjoint/fill_in_blank_valid.json',
        'polyvore-outfits/disjoint/fill_in_blank_test.json',
        'polyvore-outfits/disjoint/compatibility_valid.txt',
        'polyvore-outfits/disjoint/typespaces.p',
    ]
    for f in files:
        kaggle_dl('nguynmcch/polyvore-outfits-data', poly_dl, filename=f, unzip=True)

    out = os.path.join(RAW, 'polyvore')
    os.makedirs(out, exist_ok=True)
    for root, _, fs in os.walk(poly_dl):
        for f in fs:
            rel = os.path.relpath(os.path.join(root, f), poly_dl)
            rel = rel.replace('polyvore-outfits/', '', 1)
            dst = os.path.join(out, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.move(os.path.join(root, f), dst)
    shutil.rmtree(poly_dl, ignore_errors=True)
    mark('polyvore', 'compatibility annotations')

$ kaggle datasets download -d nguynmcch/polyvore-outfits-data -p /content/styleme_dl/poly -f polyvore-outfits/categories.csv --unzip


$ kaggle datasets download -d nguynmcch/polyvore-outfits-data -p /content/styleme_dl/poly -f polyvore-outfits/disjoint/fill_in_blank_train.json --unzip


$ kaggle datasets download -d nguynmcch/polyvore-outfits-data -p /content/styleme_dl/poly -f polyvore-outfits/disjoint/fill_in_blank_valid.json --unzip


$ kaggle datasets download -d nguynmcch/polyvore-outfits-data -p /content/styleme_dl/poly -f polyvore-outfits/disjoint/fill_in_blank_test.json --unzip


$ kaggle datasets download -d nguynmcch/polyvore-outfits-data -p /content/styleme_dl/poly -f polyvore-outfits/disjoint/compatibility_valid.txt --unzip


$ kaggle datasets download -d nguynmcch/polyvore-outfits-data -p /content/styleme_dl/poly -f polyvore-outfits/disjoint/typespaces.p --unzip


DONE polyvore: compatibility annotations


## 5. Inventory + handoff

Run this cell to confirm what 01 will consume.

In [ ]:
def human(n):
    return f'{n/1e9:.1f} GB' if n >= 1e9 else f'{n/1e6:.1f} MB'

total = 0
for root, _, files in os.walk(ROOT):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))

checks = {
    'deepfashion2': os.path.join(RAW, 'deepfashion2', 'validation.csv'),
    'fashion-product': os.path.join(RAW, 'fashion-product', 'styles.csv'),
    'fashionpedia annotations': os.path.join(RAW, 'fashionpedia', 'annotations', 'val.json'),
    'fashionpedia images': os.path.join(RAW, 'fashionpedia', 'hf', HF_PARQUET),
    'polyvore': os.path.join(RAW, 'polyvore', 'fill_in_blank_valid.json'),
    'taxonomy': os.path.join(ROOT, 'taxonomy.yaml'),
}
for name, path in checks.items():
    print(f'[{"OK" if os.path.exists(path) else "MISSING"}] {name}: {path}')
print(f'\ntotal under ROOT: {human(total)}')

# copy taxonomy next to the notebooks if the upload put it in /content/ml
tax = os.path.join(ROOT, 'taxonomy.yaml')
if not os.path.exists(tax):
    for cand in ['/content/ml/taxonomy.yaml', '/content/taxonomy.yaml']:
        if os.path.exists(cand):
            shutil.copy(cand, tax)
            print('copied taxonomy ->', tax)
            break

assert os.path.exists(os.path.join(RAW, 'deepfashion2', 'validation.csv')), 'DeepFashion2 missing'
assert os.path.exists(os.path.join(RAW, 'fashion-product', 'styles.csv')), 'Fashion Product missing'
print('\nready for 01_taxonomy_and_zero_shot.ipynb')

[OK] deepfashion2: /content/StyleMe/raw/deepfashion2/validation.csv
[OK] fashion-product: /content/StyleMe/raw/fashion-product/styles.csv
[OK] fashionpedia annotations: /content/StyleMe/raw/fashionpedia/annotations/val.json
[OK] fashionpedia images: /content/StyleMe/raw/fashionpedia/hf/data/val-00000-of-00001-f915eb4a6ec8c97e.parquet
[OK] polyvore: /content/StyleMe/raw/polyvore/fill_in_blank_valid.json
[OK] taxonomy: /content/StyleMe/taxonomy.yaml

total under ROOT: 2.0 GB

ready for 01_taxonomy_and_zero_shot.ipynb
